In [1]:
import os
from pathlib import Path
from pypdf import PdfReader

# =========================
# Rutas
# =========================


RAW_DIR = Path("data/raw_docs")
DATASET_PRACTICAS = Path("C:\\UNIVERSIDAD\\IDAL\\data\\raw_docs\\DATASET PRACTICAS")


# =========================
# Funciones de lectura
# =========================

def leer_markdown(ruta: Path) -> str:
    """Leo los archivos .md directamente."""
    with open(ruta, "r", encoding="utf-8") as f:
        return f.read()


def leer_pdf(ruta: Path) -> str:
    """Extraigo el texto de un PDF completo."""
    reader = PdfReader(str(ruta))
    texto = []

    for page in reader.pages:
        contenido = page.extract_text()
        if contenido:
            texto.append(contenido)

    return "\n".join(texto)


# =========================
# Cargar documentos
# =========================

def cargar_documentos_practicas():
    """
    Cargo todos los documentos MD y PDF de mi dataset de PRÁCTICAS.
    """
    documentos = []

    if not DATASET_PRACTICAS.exists():
        raise FileNotFoundError(f"No existe la carpeta: {DATASET_PRACTICAS}")

    print(f"📂 Cargando documentos desde: {DATASET_PRACTICAS}")

    for archivo in DATASET_PRACTICAS.rglob("*"):

        if archivo.suffix.lower() == ".md":
            texto = leer_markdown(archivo)
            tipo = "md"

        elif archivo.suffix.lower() == ".pdf":
            texto = leer_pdf(archivo)
            tipo = "pdf"

        else:
            continue  # ignorar otros formatos

        documentos.append({
            "archivo": archivo.name,
            "ruta": str(archivo),
            "dominio": "practicas",   # dominio fijo
            "tipo": tipo,
            "texto": texto
        })

    return documentos


# =========================
# Main
# =========================

if __name__ == "__main__":
    documentos = cargar_documentos_practicas()

    print(f"\nDocumentos cargados: {len(documentos)}")

    for doc in documentos[:5]:
        print("-" * 50)
        print("Archivo:", doc["archivo"])
        print("Dominio:", doc["dominio"])
        print("Tipo:", doc["tipo"])
        print("Texto inicial:")
        print(doc["texto"][:500])


In [ ]:
from pathlib import Path
import json
import re
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =========================
# Rutas
# =========================

RAW_DIR = Path("C:\\UNIVERSIDAD\\IDAL\\data\\raw_docs\\DATASET PRACTICAS")
OUTPUT_FILE = Path("data/chunks_semanticos.json")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# =========================
# Lectura de documentos
# =========================

def leer_markdown(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        return f.read()


def leer_pdf(ruta):
    reader = PdfReader(str(ruta))
    paginas = []

    for i, page in enumerate(reader.pages):
        texto = page.extract_text() or ""
        paginas.append({
            "pagina": i + 1,
            "texto": texto
        })

    return paginas

# =========================
# Detección de títulos
# =========================

def limpiar_titulo(linea):
    linea = linea.strip()
    linea = re.sub(r"^#{1,6}\s*", "", linea)
    linea = re.sub(r"^[\*\-\–•]\s*", "", linea)
    linea = linea.replace("**", "").replace("__", "").replace("`", "")
    return linea.strip()


def es_titulo(linea):
    linea = linea.strip()

    if not linea:
        return False

    if re.fullmatch(r"[-_*]{3,}", linea):
        return False

    # Títulos Markdown reales
    if re.match(r"^#{1,6}\s+", linea):
        return True

    # Títulos en negrita tipo "**Título**"
    if re.match(r"^\*\*[^*]{3,120}\*\*:?$", linea):
        return True

    # Títulos tipo "4.1 Algo"
    if re.match(r"^\d+[\.\)]\s+[A-ZÁÉÍÓÚÜÑ]", linea):
        return True

    # Artículos legales
    if re.match(r"^(Artículo|Articulo|ARTÍCULO|ARTICLE|Article)\s+\d+", linea):
        return True

    # Títulos en mayúsculas
    if re.match(r"^([IVXLCDM]+\.\s+)?[A-ZÁÉÍÓÚÜÑ0-9\s,/:()-]{5,140}$", linea):
        return True

    return False



def dividir_por_titulos(texto):
    lineas = texto.splitlines()

    secciones = []
    titulo_actual = "Inicio del documento"
    contenido_actual = []

    for linea in lineas:
        if es_titulo(linea):
            if contenido_actual:
                secciones.append({
                    "titulo": titulo_actual,
                    "texto": "\n".join(contenido_actual).strip()
                })

            titulo_actual = limpiar_titulo(linea)
            contenido_actual = [linea]
        else:
            contenido_actual.append(linea)

    if contenido_actual:
        secciones.append({
            "titulo": titulo_actual,
            "texto": "\n".join(contenido_actual).strip()
        })

    return secciones

